In [2]:
# day 4 - defining who is "churned" and who isn't
# i'm pretending it's april 1, 2018 and i'm trying to decide who to send retention emails to.

import pandas as pd
from pathlib import Path

# i saved cleaned data yesterday as parquet files - loading them now
# this is way faster than re-running all of day 3
PROCESSED_DIR = Path("../data/processed")
orders_with_customer = pd.read_parquet(PROCESSED_DIR / "orders_with_customer.parquet")

print(f"loaded {len(orders_with_customer):,} delivered orders")
print(f"unique customers: {orders_with_customer['customer_unique_id'].nunique():,}")
print()

# defining the three key dates
# CUTOFF = "today" in our pretend scenario. all features must come from before this date.
# OBSERVATION_START = how far back to look to find "recently active" customers
# FORWARD_END = how far forward to look to see if they came back

CUTOFF = pd.Timestamp("2018-04-01")
OBSERVATION_START = CUTOFF - pd.DateOffset(months=6)   # 2017-10-01
FORWARD_END = CUTOFF + pd.DateOffset(months=3)         # 2018-07-01

print(f"observation window: {OBSERVATION_START.date()} to {CUTOFF.date()}")
print(f"cutoff date (today): {CUTOFF.date()}")
print(f"forward window:     {CUTOFF.date()} to {FORWARD_END.date()}")

loaded 96,478 delivered orders
unique customers: 93,358

observation window: 2017-10-01 to 2018-04-01
cutoff date (today): 2018-04-01
forward window:     2018-04-01 to 2018-07-01


In [3]:
# step 1: find every customer who placed an order in the observation window
# (oct 1, 2017 -> apr 1, 2018). these are my "recently active" customers.
# only these people are eligible for the model - everyone else either bought too long ago
# or hasn't bought yet at all.

# filter orders to ONLY the observation window
# note the < (not <=) on the right side - we don't want to include orders on the cutoff date itself
# because the cutoff is "today" and we're predicting the future
observation_orders = orders_with_customer[
    (orders_with_customer["order_purchase_timestamp"] >= OBSERVATION_START) &
    (orders_with_customer["order_purchase_timestamp"] < CUTOFF)
]

print(f"orders in observation window: {len(observation_orders):,}")

# unique customers who placed at least one order in the window = eligible
eligible_customers = observation_orders["customer_unique_id"].unique()
print(f"eligible customers (placed >=1 order in window): {len(eligible_customers):,}")

# turning this into a dataframe so it's easier to work with
eligible_df = pd.DataFrame({"customer_unique_id": eligible_customers})
print()
print("first 5 eligible customer ids:")
print(eligible_df.head())

orders in observation window: 37,907
eligible customers (placed >=1 order in window): 37,042

first 5 eligible customer ids:
                 customer_unique_id
0  7c396fd4830fd04220f754e42b4e5bff
1  7c142cf63193a1473d2e66489a9ae977
2  72632f0f9dd73dfee390c9b22eb56dd6
3  04cf8185c71090d28baa4407b2e6d600
4  6e26bbeaa107ec34112c64e1ee31c0f5


In [4]:
# step 2: now look forward into the april-july 2018 window.
# for each eligible customer, did they buy anything in this future window?
# - yes -> retained (label = 0)
# - no  -> churned (label = 1)

# filter orders to the forward window
forward_orders = orders_with_customer[
    (orders_with_customer["order_purchase_timestamp"] >= CUTOFF) &
    (orders_with_customer["order_purchase_timestamp"] < FORWARD_END)
]

print(f"orders in forward window: {len(forward_orders):,}")

# unique customers who bought something in the forward window = "they came back"
returners = set(forward_orders["customer_unique_id"].unique())
print(f"customers who came back in forward window: {len(returners):,}")
print()

# now label each eligible customer
# 1 if they're NOT in the returners set (= churned), 0 if they are (= retained)
eligible_df["churned"] = (~eligible_df["customer_unique_id"].isin(returners)).astype(int)

print("label distribution:")
print(eligible_df["churned"].value_counts())
print()

# print as percentages for clarity
print("label distribution (percentages):")
churn_pct = eligible_df["churned"].value_counts(normalize=True) * 100
print(f"  churned (1):  {churn_pct[1]:.2f}%")
print(f"  retained (0): {churn_pct[0]:.2f}%")

orders in forward window: 19,646
customers who came back in forward window: 19,391

label distribution:
churned
1    36750
0      292
Name: count, dtype: int64

label distribution (percentages):
  churned (1):  99.21%
  retained (0): 0.79%


In [5]:
# the first plan didn't work - only 0.79% retained in the forward window.
# too imbalanced to model usefully.
#
# new plan: narrow the eligible population to customers who already showed they're "repeat buyers"
# (placed 2+ orders in the observation window). these customers actually have a chance of returning.
# also widening the forward window from 3 to 6 months.
#
# business framing: "of customers who showed repeat-buying behavior, which ones will continue?"
# this is a more useful question for the business than "of one-time buyers, will they come back?"

# updating the forward end date - now 6 months instead of 3
FORWARD_END = CUTOFF + pd.DateOffset(months=6)   # 2018-10-01
print(f"new forward window: {CUTOFF.date()} to {FORWARD_END.date()}")
print()

# count orders per customer IN THE OBSERVATION WINDOW
obs_order_counts = observation_orders.groupby("customer_unique_id").size()
print("distribution of orders per customer in observation window:")
print(obs_order_counts.value_counts().sort_index().head(10))
print()

# narrow eligible to customers with 2+ orders in observation window
repeat_eligible = obs_order_counts[obs_order_counts >= 2].index
print(f"customers with 2+ orders in observation window: {len(repeat_eligible):,}")

# rebuild the forward orders with new end date
forward_orders = orders_with_customer[
    (orders_with_customer["order_purchase_timestamp"] >= CUTOFF) &
    (orders_with_customer["order_purchase_timestamp"] < FORWARD_END)
]
returners = set(forward_orders["customer_unique_id"].unique())
print(f"customers who came back in new forward window: {len(returners):,}")
print()

# build the new label dataframe
eligible_df = pd.DataFrame({"customer_unique_id": repeat_eligible})
eligible_df["churned"] = (~eligible_df["customer_unique_id"].isin(returners)).astype(int)

print("new label distribution:")
print(eligible_df["churned"].value_counts())
print()

churn_pct = eligible_df["churned"].value_counts(normalize=True) * 100
print("new label distribution (percentages):")
print(f"  churned (1):  {churn_pct[1]:.2f}%")
print(f"  retained (0): {churn_pct[0]:.2f}%")

new forward window: 2018-04-01 to 2018-10-01

distribution of orders per customer in observation window:
1    36232
2      775
3       25
4        5
5        3
7        1
8        1
Name: count, dtype: int64

customers with 2+ orders in observation window: 810
customers who came back in new forward window: 31,662

new label distribution:
churned
1    774
0     36
Name: count, dtype: int64

new label distribution (percentages):
  churned (1):  95.56%
  retained (0): 4.44%


## Decision: Pivot from churn prediction to delivery delay prediction

During exploration, I discovered that the Olist dataset has an extreme one-time-buyer pattern: 97% of all customers, and 95% of even "recently active" customers, never place another order within a reasonable forward window. Every churn definition I tried produced class balance worse than 95/5 — too imbalanced to train a meaningful model on.

Rather than force a poor-fit model, I pivoted the target variable to predicting **delivery delay** (will this order arrive after its estimated delivery date?). This decision is documented in this notebook for transparency.

**Why this pivot makes sense:**
- Delivery delay has ~8% positive class — challenging but modelable
- The business value is clear: predicting delays at order time lets Olist proactively notify customers, expedite shipping, or flag unreliable sellers
- The features in the data (buyer/seller geography, product dimensions, freight cost, time of year) directly relate to delivery time
- All Day 1-3 work (data cleaning, table joins, processed parquet files) carries over unchanged

The project will retain the name `churn-radar` for GitHub continuity but the README and resume bullets will accurately describe the delivery-delay prediction system that was actually built.